# ResNet-20 (FP32) on CIFAR-10 — Kaggle training notebook
Output: `/kaggle/working/resnet20_fp32.pt` (final-epoch weights, no test-set checkpoint selection).

In [ ]:
import os, time, json, random, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T

CFG = dict(seed=0, epochs=160, bs=128, lr=0.1, wd=5e-4, momentum=0.9,
           workers=2, amp=True, out_dir="/kaggle/working")
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"])
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev == "cuda" else "")

## Data
Uses an attached Kaggle dataset containing `cifar-10-batches-py` if found; otherwise downloads (needs Internet ON).

In [ ]:
def find_root():
    hits = glob.glob("/kaggle/input/**/cifar-10-batches-py", recursive=True)
    return (os.path.dirname(hits[0]), False) if hits else ("/kaggle/working/data", True)

root, dl = find_root()
print("data root:", root, "| download:", dl)

tf_train = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
tf_test = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
train_set = torchvision.datasets.CIFAR10(root, train=True, download=dl, transform=tf_train)
test_set = torchvision.datasets.CIFAR10(root, train=False, download=dl, transform=tf_test)
train_loader = torch.utils.data.DataLoader(train_set, CFG["bs"], shuffle=True,
                                           num_workers=CFG["workers"], pin_memory=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_set, 500, shuffle=False, num_workers=CFG["workers"])

## Model: CIFAR ResNet-20 (3 stages x 3 blocks, 16/32/64 ch, option-A shortcuts, ~0.27M params)

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.pad = (cout - cin) if (stride != 1 or cin != cout) else 0
        self.stride = stride
    def shortcut(self, x):
        if self.pad == 0: return x
        x = x[:, :, ::self.stride, ::self.stride]
        return F.pad(x, (0, 0, 0, 0, self.pad // 2, self.pad // 2))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNetCifar(nn.Module):
    def __init__(self, n=3, num_classes=10):   # depth = 6n + 2  (n=3 -> ResNet-20)
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False); self.bn1 = nn.BatchNorm2d(16)
        layers, cin = [], 16
        for cout, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(n):
                layers.append(Block(cin, cout, stride if i == 0 else 1)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(64, num_classes)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)): nn.init.kaiming_normal_(m.weight)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layers(x)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

model = ResNetCifar(n=3).to(dev)
print("params:", sum(p.numel() for p in model.parameters()))

## Train

In [ ]:
opt = torch.optim.SGD(model.parameters(), lr=CFG["lr"], momentum=CFG["momentum"],
                      weight_decay=CFG["wd"], nesterov=True)
steps = CFG["epochs"] * len(train_loader)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=steps,
                                            pct_start=0.05, anneal_strategy="cos")
scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"] and dev == "cuda")

@torch.no_grad()
def evaluate():
    model.eval(); correct = 0
    for x, y in test_loader:
        correct += (model(x.to(dev)).argmax(1).cpu() == y).sum().item()
    return 100 * correct / len(test_set)

log = []
for ep in range(CFG["epochs"]):
    model.train(); t0 = time.time(); tot = 0.0
    for x, y in train_loader:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.autocast(dev, enabled=CFG["amp"] and dev == "cuda"):
            loss = F.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    acc = evaluate() if (ep % 10 == 9 or ep == CFG["epochs"] - 1) else None
    log.append(dict(epoch=ep + 1, loss=tot / len(train_loader), test_acc=acc))
    print(f"ep {ep+1:3d} loss {log[-1]['loss']:.4f} acc {acc} ({time.time()-t0:.1f}s)")

## Save (final epoch; BN stats and normalization constants included)

In [ ]:
ckpt = dict(state_dict=model.state_dict(), cfg=CFG, mean=MEAN, std=STD, final_test_acc=log[-1]["test_acc"])
torch.save(ckpt, f"{CFG['out_dir']}/resnet20_fp32.pt")
json.dump(log, open(f"{CFG['out_dir']}/train_log.json", "w"))
print("saved. final test acc:", log[-1]["test_acc"])